In [3]:
from docx import Document
import os
import jsonlines
import pandas as pd

In [215]:
# 定义一个函数来处理段落
def process_paragraph(para, file_name, headings, doc):
       # 替换list连接符
    # headings = '->'.join(headings)
    global module_count
    if para.text.strip():
        module_count += 1
        blocks.append({
            'module_id': module_count,
            'type': 'text',
            'content': para.text.strip(),
            'file_name': file_name,
            'headings': headings
        })
# 定义一个函数来处理标题
def process_headings(para, file_name, headings, doc):
    # 替换list连接符
    # headings = '->'.join(headings)
    global module_count
    if para.text.strip():
        module_count += 1
        blocks.append({
            'module_id': module_count,
            'type': 'headings',
            'content': para.text.strip(),
            'file_name': file_name,
            'headings': headings
        })        
# 判断段落是否为标题
def is_heading(para,current_headings):
    if 'Heading' in para.style.name:  # 检查段落是否为标题
        level = int(para.style.name[-1])
        current_headings = current_headings[:level-1] + [para.text]
        return 'Heading',current_headings
    return 'text',current_headings

# 将各级标题构建层级关系三元组保存在output_headings_triples_path
def headings_triples(df,file_name):
    # 处理标题层级关系
    df_headings = df[df.type=='headings']
    # 将列表展开为多列
    df_head = df_headings['headings'].apply(pd.Series)
    # 增加标题列
    df_head = pd.concat([df_headings[['file_name']],df_head],axis=1)
    # df_head.columns = [str(i) for i in range(len(df_head.columns))]
    # 将 NaN 替换为空字符串 ''
    df_head = df_head.fillna('')
    # 初始化用于存储三元组的列表
    triples = []
    # 更新非空单元格 aij 为 aij-1 + aij
    for idx, item in enumerate(df_head.columns[1:], start=1):
        prev_item = df_head.columns[idx - 1]  # 获取前一列的列名
        for i in df_head.index:
            current_value = df_head.at[i, item]
            if isinstance(current_value, str) and current_value.strip() != '':
                previous_value = df_head.at[i, prev_item]
                new_value = previous_value + '-' + current_value
                df_head.at[i, item] = new_value
                # 存储三元组 (头实体, 尾实体)
                triple = [previous_value.strip(), '包含', new_value]
                if triple not in triples:
                    triples.append(triple)
    # print(df_head)
    # {"triples": [triples]}
    # output_headings_triples_path = "/workspace/term_basev2_1203/output/" + file_name +"_Triples_1225.json"
    # # 检查文件是否存在
    # if os.path.exists(output_headings_triples_path):
    #     try:
    #         # 尝试删除文件
    #         os.remove(output_headings_triples_path)
    #         print(f"文件 '{output_headings_triples_path}' 已成功删除。")
    #     except OSError as e:
    #         # 如果删除失败，打印错误信息
    #         print(f"无法删除文件 '{output_headings_triples_path}': {e}")
    # with open(output_headings_triples_path,'a') as f:
    #     print(f"文件 '{output_headings_triples_path}' 正在写入三元组。")
    #     writer = jsonlines.Writer(f)
    #     writer.write({"triples": [triples]})
    return triples

def merge_chunk(df, file_name):    
    merged_blocks = []
    current_merged_text = ''
    current_tokens = 0
    chunk_triples = []
    print("Merging blocks...")
    
    # 遍历 DataFrame 中的每一行
    for index, row in df.iterrows():
        if row['type'] == 'text':
            para_text = row['content'].replace(' ','')
            tokens = len(para_text)
            if current_tokens + tokens <= 512:
                # print('current_tokens',current_tokens)
                current_merged_text += para_text + '\n'
                current_tokens += tokens
                # print('current_tokens',current_tokens)
                current_headings = row['headings']
            else:
                # 如果超过 512 个字符，则保存当前合并的段落
                current_headings = '-'.join(current_headings)
                headings = f"{file_name}-{current_headings}".strip()
                merged_blocks.append({
                    'type': 'text',
                    'position': len(merged_blocks),
                    'content': current_merged_text,
                    'headings': headings
                })
                chunk_triples.append([headings, '包含', current_merged_text])
                # 开始新的合并段落
                current_merged_text = para_text + '\n'
                current_tokens = tokens
                current_headings = row['headings']
        elif row['type'] == 'headings':
            if current_tokens > 15:
                # 如果当前有未保存的合并段落，则保存
                current_headings = '-'.join(current_headings)
                headings = f"{file_name}-{current_headings}".strip()
                merged_blocks.append({
                    'type': 'text',
                    'position': len(merged_blocks),
                    'content': current_merged_text,
                    'headings': headings
                })
                chunk_triples.append([headings, '包含', current_merged_text])
            current_merged_text = ''
            current_tokens = 0
            current_headings = row['headings']
    
    # 处理最后一个合并段落
    if current_tokens > 15:
        current_headings = '-'.join(current_headings)
        headings = f"{file_name}-{current_headings}".strip()
        merged_blocks.append({
            'type': 'text',
            'position': len(merged_blocks),
            'content': current_merged_text,
            'headings': headings
        })
        chunk_triples.append([headings, '包含', current_merged_text])
    # 将结果列表转换为 DataFrame
    merged_df = pd.DataFrame(merged_blocks)
    # # 输出 DataFrame
    # print("Final Merged DataFrame:")
    # print(merged_df)
    
    # # 保存 DataFrame 到新的 CSV 文件
    merged_output_module = '../temp_data/' + file_name+'_chunk_1225.csv'
    merged_df.to_csv(merged_output_module, index=False)
    print(f"Saved final merged DataFrame to: {merged_output_module}")
    return chunk_triples

In [221]:
# main
global module_count, blocks
module_count = 0
blocks = []
current_headings = []
file_path = '/workspace/term_basev2_1203/data/中国电信10000号客户服务数据运营规范.docx'

if not os.path.exists(file_path):
    raise FileNotFoundError(f"文件未找到：{file_path}")
# 加载 DOCX 文件
doc = Document(file_path)
file_name = os.path.splitext(os.path.basename(file_path))[0]
headings = []
# current_head = file_name
# 遍历文档中的所有元素
for element in doc.element.body:
    # print('element.tag:',element.tag)
    if element.tag.endswith('p'):
        for para in doc.paragraphs:
            if para._element is element:
                temp,current_headings = is_heading(para,current_headings)
                # current_head =file_name + '-' +  '-'.join(current_headings)
                # print('current_head',current_head)
                if temp=='Heading':
                    # print('Heading:',current_headings)
                    
                    process_headings(para, file_name, current_headings, doc)
                else:
                    # headings.append(para.text.strip())
                    process_paragraph(para, file_name, current_headings, doc)
                break
    elif element.tag.endswith('tbl'):
        print('table')
        for table in doc.tables:
            if table._element is element:
                process_table(table, file_name, current_headings, doc,blocks)
                break
# 将结果列表转换为 DataFrame
df = pd.DataFrame(blocks)
# 将各级标题构建层级关系三元组保存在output_headings_triples_path
title_triples = headings_triples(df,file_name)
print('title_triples:',title_triples)
chunk_triples = merge_chunk(df, file_name)
print('chunk_triples:',chunk_triples)
title_chunk_triples = title_triples + chunk_triples

output_headings_triples_path = "/workspace/term_basev2_1203/output/" + file_name +"_Triples_1226v2.json"
# 检查文件是否存在
if os.path.exists(output_headings_triples_path):
    try:
        # 尝试删除文件
        os.remove(output_headings_triples_path)
        print(f"文件 '{output_headings_triples_path}' 已成功删除。")
    except OSError as e:
        # 如果删除失败，打印错误信息
        print(f"无法删除文件 '{output_headings_triples_path}': {e}")
with open(output_headings_triples_path,'a') as f:
    print(f"文件 '{output_headings_triples_path}' 正在写入三元组。")
    writer = jsonlines.Writer(f)
    writer.write({"last_triples": [title_chunk_triples]})

title_triples: [['中国电信10000号客户服务数据运营规范', '包含', '中国电信10000号客户服务数据运营规范-文档说明'], ['中国电信10000号客户服务数据运营规范', '包含', '中国电信10000号客户服务数据运营规范-职责分工'], ['中国电信10000号客户服务数据运营规范', '包含', '中国电信10000号客户服务数据运营规范-10000号服务数据运营原则'], ['中国电信10000号客户服务数据运营规范', '包含', '中国电信10000号客户服务数据运营规范-数据分类'], ['中国电信10000号客户服务数据运营规范', '包含', '中国电信10000号客户服务数据运营规范-话务数据分类及标签'], ['中国电信10000号客户服务数据运营规范', '包含', '中国电信10000号客户服务数据运营规范-工单数据分类及标签'], ['中国电信10000号客户服务数据运营规范', '包含', '中国电信10000号客户服务数据运营规范-服务数据采集'], ['中国电信10000号客户服务数据运营规范', '包含', '中国电信10000号客户服务数据运营规范-服务数据迭代'], ['中国电信10000号客户服务数据运营规范', '包含', '中国电信10000号客户服务数据运营规范-服务数据全生命周期运营管理'], ['中国电信10000号客户服务数据运营规范-文档说明', '包含', '中国电信10000号客户服务数据运营规范-文档说明-编制说明'], ['中国电信10000号客户服务数据运营规范-文档说明', '包含', '中国电信10000号客户服务数据运营规范-文档说明-适用范围'], ['中国电信10000号客户服务数据运营规范-文档说明', '包含', '中国电信10000号客户服务数据运营规范-文档说明-解释权'], ['中国电信10000号客户服务数据运营规范-文档说明', '包含', '中国电信10000号客户服务数据运营规范-文档说明-起草单位'], ['中国电信10000号客户服务数据运营规范-文档说明', '包含', '中国电信10000号客户服务数据运营规范-文档说明-编制人员'], ['中国电信10000号客户服务数据运营规范-文档说明', '包含', '中国电信10000号客